# ML-07: Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahmooudEssam/FlyRank-Intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order**, each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule in plain words:** among page-2 pages with real visibility, review first the pages losing the most clicks compared with pages at a similar position.

- **Score:** `missed_clicks`, the clicks a page would get at its peers' CTR minus the clicks it really got.
- **Reason code:** `LOW_CTR_FOR_POSITION`
- **Action label:** `Rewrite title and meta description`

**Data:** the starter file `content_refresh_anonymized.csv`, one row per page.

**Decision moment:** 30 days before the export. The rule may only use the 30 days before that moment (`*_prev_30d`). The most recent 30 days are kept for checking.

**Who is in the slice:** average position above 10 and up to 20, at least 300 impressions in the input window, and no update in the last 30 days, because an edit during the check window would blur the result.

**How I check signals:** `impressions_fell` is 1 when impressions in the last 30 days dropped more than 20% from the 30 days before. This file can't show whether a page reached page 1, so this is a different outcome from my W02 label. It is used for checking only and never goes into the score.

In [1]:
import json
from pathlib import Path

import pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
CSV_URL = "https://raw.githubusercontent.com/MahmooudEssam/FlyRank-Intern/main/" + CSV

# Find the repo root on this computer. In Colab, read the file from GitHub instead.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / CSV).exists()), None)
data = pd.read_csv(root / CSV if root else CSV_URL)
root = root or Path.cwd()

pages = data[
    data["avg_position"].between(10, 20, inclusive="right")
    & (data["impressions_prev_30d"] >= 300)
    & (data["days_since_last_update"] >= 30)
].copy()

# The outcome, for checking only
pages["impressions_fell"] = (pages["impressions_last_30d"] < 0.8 * pages["impressions_prev_30d"]).astype(int)

# One client is very large, so every table also shows the result without it
largest_client = pages["client_id"].value_counts().index[0]


def bucket_table(bucket):
    others = pages[pages["client_id"] != largest_client]
    table = pages.groupby(bucket, observed=False).agg(
        n=("impressions_fell", "size"),
        clients=("client_id", "nunique"),
        fell_rate=("impressions_fell", "mean"),
    )
    table["n_without_largest"] = others.groupby(bucket, observed=False).size()
    table["fell_rate_without_largest"] = others.groupby(bucket, observed=False)["impressions_fell"].mean()
    return table.round(3)


base_rate = pages["impressions_fell"].mean()
print(f"Pages in the slice: {len(pages):,} from {pages['client_id'].nunique()} clients")
print(f"Largest client holds {(pages['client_id'] == largest_client).mean():.1%} of them")
print(f"Base rate, impressions fell: {base_rate:.1%}")

Pages in the slice: 1,241 from 14 clients
Largest client holds 62.0% of them
Base rate, impressions fell: 62.9%


### Signal 1: staleness (behind the refresh flags)

**My expectation:** pages left alone for longer lose more impressions.

In [2]:
pages["days_since_update_bucket"] = pd.cut(pages["days_since_last_update"], [29, 103, 104, 180, 1000],
                                           labels=["30-103", "exactly 104", "105-180", "181+"])
bucket_table("days_since_update_bucket")

,n,clients,fell_rate,n_without_largest,fell_rate_without_largest
days_since_update_bucket,,,,,
30-103,14,6,0.571,12,0.583
exactly 104,1206,10,0.624,438,0.664
105-180,15,3,0.933,15,0.933
181+,6,1,1.000,6,1.000


**Verdict: MIXED.** The oldest buckets fell the most, but they hold only 21 pages from very few clients. About 97% of the slice was last updated exactly 104 days ago, which looks like one batch update, so staleness can barely separate pages here. I keep it out of the score.

### Signal 2: CTR compared with position (behind the CTR-fix logic)

**My expectation:** pages with a lower CTR than their position peers lose more impressions. Peers are pages in the same two-position band, and their CTR is total clicks divided by total impressions.

In [3]:
pages["position_band"] = pd.cut(pages["avg_position"], [10, 12, 14, 16, 18, 20])
band_totals = pages.groupby("position_band", observed=True)[["clicks_prev_30d", "impressions_prev_30d"]].transform("sum")

pages["peer_ctr"] = band_totals["clicks_prev_30d"] / band_totals["impressions_prev_30d"]
pages["ctr_prev"] = pages["clicks_prev_30d"] / pages["impressions_prev_30d"]
pages["ctr_vs_peers"] = pd.cut(pages["ctr_prev"] / pages["peer_ctr"], [-1, 0, 0.5, 1, 2, float("inf")],
                               labels=["zero clicks", "under half", "half to equal", "1x to 2x", "over 2x"])
bucket_table("ctr_vs_peers")

,n,clients,fell_rate,n_without_largest,fell_rate_without_largest
ctr_vs_peers,,,,,
zero clicks,362,12,0.754,224,0.719
under half,309,9,0.634,123,0.618
half to equal,231,9,0.602,82,0.695
1x to 2x,218,8,0.505,32,0.531
over 2x,121,7,0.521,10,0.700


**Verdict: MIXED.** Across all pages the pattern fits: 75.4% of zero-click pages fell, against about 51% of pages at or above their peers. Without the largest client it gets bumpy: zero-click pages are still high at 71.9%, but "half to equal" is 69.5% and "over 2x" is 70.0% on only 10 pages. I still use this signal because it points to a clear fix, but I treat it as directional.

## 2. Build the ranked queue (writes the CSV)

Expected clicks = peer CTR x impressions. The score is expected clicks minus actual clicks. Only pages below their peers enter the queue, and ties are broken by `content_id` so the order never changes.

In [4]:
pages["expected_clicks"] = pages["peer_ctr"] * pages["impressions_prev_30d"]
pages["score"] = pages["expected_clicks"] - pages["clicks_prev_30d"]

queue = pages[pages["score"] > 0].sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)
queue["reason_code"] = "LOW_CTR_FOR_POSITION"
queue["action"] = "Rewrite title and meta description"

output_dir = root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
columns = ["rank", "content_id", "score", "reason_code", "action", "avg_position",
           "impressions_prev_30d", "clicks_prev_30d", "expected_clicks"]
queue[columns].round(2).to_csv(output_dir / "baseline_action_score.csv", index=False)

# How often did the top of the queue really lose impressions?
metrics = {"pages_in_slice": len(pages), "pages_in_queue": len(queue), "base_rate": round(base_rate, 3)}
for k in (10, 50, 100):
    metrics[f"precision_at_{k}"] = round(queue["impressions_fell"].head(k).mean(), 3)
(output_dir / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))

print(json.dumps(metrics, indent=2))

{
  "pages_in_slice": 1241,
  "pages_in_queue": 902,
  "base_rate": 0.629,
  "precision_at_10": 0.4,
  "precision_at_50": 0.7,
  "precision_at_100": 0.66
}


## 3. Top-10 review

Each page gets the same action because this is a one-rule baseline. The "why" comes from the data, and I wrote each "wrong if" after looking at that page.

In [5]:
wrong_if = {
    "content_eb366e871254": "It ranks for many loosely related queries, so a new title would not fix a broad, low-intent audience.",
    "content_c5063073d048": "A few big queries show search features that take the clicks no matter what the title says.",
    "content_a69b8a33d165": "Position 10.1 hides a mix of page-1 and page-2 days, so its peer group is a poor match.",
    "content_47f14fc1ca93": "Its commercial searchers want product or pricing pages, which an article cannot satisfy.",
    "content_00202ac57009": "Its intent is not recorded. If people search for a specific brand or site, no title will win them.",
    "content_c22c1dd7122d": "The topic is seasonal and the input window was simply a quiet month for clicks.",
    "content_de57bb97bbf1": "A site-wide template writes its titles, so the fix belongs on the template and not on this page.",
    "content_cf56e2e2e282": "The content is out of date after 194 days without an update, so it needs a refresh, not a new title.",
    "content_d91092100626": "A sibling page from the same site targets the same queries, so merging them is the better fix.",
    "content_758db544d84f": "The gap is normal variation: it has the smallest gap against peers in this top 10.",
}

top_10 = queue.head(10).copy()
top_10["why"] = [
    f"Position {r.avg_position:.1f}: {r.clicks_prev_30d:,} clicks vs about {r.expected_clicks:,.0f} expected from {r.impressions_prev_30d:,} impressions"
    for r in top_10.itertuples()
]
top_10["wrong_if"] = top_10["content_id"].map(wrong_if)
assert top_10["wrong_if"].notna().all()  # a new top 10 needs a new review

pd.set_option("display.max_colwidth", None)
top_10[["rank", "content_id", "action", "reason_code", "why", "wrong_if"]].style.hide(axis="index")

rank,content_id,action,reason_code,why,wrong_if
1,content_eb366e871254,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 16.6: 108 clicks vs about 255 expected from 58,717 impressions","It ranks for many loosely related queries, so a new title would not fix a broad, low-intent audience."
2,content_c5063073d048,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 12.5: 189 clicks vs about 329 expected from 65,696 impressions",A few big queries show search features that take the clicks no matter what the title says.
3,content_a69b8a33d165,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 10.1: 20 clicks vs about 117 expected from 21,773 impressions","Position 10.1 hides a mix of page-1 and page-2 days, so its peer group is a poor match."
4,content_47f14fc1ca93,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 17.2: 36 clicks vs about 130 expected from 29,853 impressions","Its commercial searchers want product or pricing pages, which an article cannot satisfy."
5,content_00202ac57009,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 18.0: 17 clicks vs about 101 expected from 23,334 impressions","Its intent is not recorded. If people search for a specific brand or site, no title will win them."
6,content_c22c1dd7122d,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 14.9: 50 clicks vs about 129 expected from 27,071 impressions",The topic is seasonal and the input window was simply a quiet month for clicks.
7,content_de57bb97bbf1,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 13.0: 63 clicks vs about 137 expected from 27,345 impressions","A site-wide template writes its titles, so the fix belongs on the template and not on this page."
8,content_cf56e2e2e282,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 19.7: 28 clicks vs about 97 expected from 26,791 impressions","The content is out of date after 194 days without an update, so it needs a refresh, not a new title."
9,content_d91092100626,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 16.7: 40 clicks vs about 107 expected from 24,626 impressions","A sibling page from the same site targets the same queries, so merging them is the better fix."
10,content_758db544d84f,Rewrite title and meta description,LOW_CTR_FOR_POSITION,"Position 13.8: 164 clicks vs about 230 expected from 45,777 impressions",The gap is normal variation: it has the smallest gap against peers in this top 10.


## 4. Weak picks + leakage check

**Weak picks I found:**

- **Rank 3** sits at position 10.1, right on the edge of page 1, so comparing it with page-2 peers is shaky.
- **Rank 8** has not been updated for 194 days and sits at position 19.7. It may need a content refresh more than a new title.
- **Rank 10** has the smallest gap against its peers, so it may not be underperforming at all.
- **The top 10 is too concentrated.** The score rewards big pages, so nearly all of it comes from one client (numbers below). An editor would want a cap per client.
- **The very top is not better than average.** Precision@10 is below the base rate, while precision@50 is a little above it. This rule is a starting point to beat.

**Leakage check:** the score uses no last-30-day columns, no `trend_pct` or `trend_direction`, and no outcome. The code below confirms it.

**One overlap remains:** `avg_position` covers the full 90 days, which includes the check window. This file has no position for the earlier 30 days alone. It only chooses the slice and the peer band, but it can still slightly flatter these checks, so the numbers are directional. A warehouse version could remove this overlap.

In [6]:
score_inputs = {"avg_position", "impressions_prev_30d", "clicks_prev_30d"}
not_allowed = {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_90d", "clicks_90d",
               "ctr", "trend_pct", "trend_direction", "impressions_fell"}
assert not score_inputs & not_allowed
print("Score inputs:", sorted(score_inputs))
print("None of them are last-30-day, trend, or outcome columns")

top_10_from_largest = (queue["client_id"].head(10) == largest_client).sum()
top_50_from_largest = (queue["client_id"].head(50) == largest_client).sum()
print(f"Largest client: {top_10_from_largest} of the top 10, {top_50_from_largest} of the top 50")

Score inputs: ['avg_position', 'clicks_prev_30d', 'impressions_prev_30d']
None of them are last-30-day, trend, or outcome columns
Largest client: 9 of the top 10, 40 of the top 50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.